# 多阶段细胞生物学视觉推理：从最小参考文献到独立假说

本 Notebook 是一份可逐格运行的教学教程。它完成两件事：

1. 把上一阶段找到的最小参考文献重新整理成可复用的数据包，包括论文原文、全文文本、每张原始图片、专属图注，以及单独封存的目标论文假说。
2. 按 Stage 1（四个视觉专家独立描述）→ Stage 2（单图融合）→ 单篇全局 Context → Stage 3（多文献联合生成假说）执行模型调用。

## 最重要的防泄漏原则

目标论文的真实假说仅保存到 `sealed_target/`，用于最后人工评价。Stage 1、Stage 2、Context 和 Stage 3 的代码都不会读取该目录。否则模型可能只是复述目标答案，不能算独立推理。

代码使用英文；教学解释使用中文；发送给模型的 Prompt 使用英文。

## 第0课：理解最终目录

运行数据准备后，将生成：

```text
LLM_Reasoning/data/test/
├── dataset_manifest.json                 # 全部素材的总索引
├── provenance/                           # 上一步结果的只读副本
├── references/
│   ├── ref1_CR22_PMC7612910/
│   │   ├── source/article.xml             # PMC JATS原文
│   │   ├── source/full_text.txt           # 方便LLM读取的全文
│   │   ├── source/article.json            # 论文元数据
│   │   ├── figures/                       # 原始图片
│   │   ├── figures_manifest.json          # 图片—图注—来源映射
│   │   └── derived/
│   │       ├── context.json               # 每篇论文唯一全局Context
│   │       └── observations/<figure_id>/  # 四专家描述与融合Observation
│   ├── ref2_.../
│   └── ref3_.../
├── reasoning_inputs/stage3_input.json     # Stage 3唯一允许读取的输入
├── reasoning_outputs/stage3_hypotheses.json
└── sealed_target/target_hypothesis.json   # 封存答案，禁止进入Stage 1–3
```

## 第1课：配置路径和运行开关

In [1]:
from pathlib import Path
import base64
import hashlib
import html
import itertools
import json
import mimetypes
import os
import re
import shutil
import tarfile
import time
import xml.etree.ElementTree as ET

import httpx

PROJECT_ROOT = Path(r'C:/Users/sxx/Desktop/codex/barchmark-m-7.30')
REASONING_ROOT = PROJECT_ROOT / 'LLM_Reasoning'
DATA_ROOT = REASONING_ROOT / 'data' / 'test'
SOURCE_RESULT = PROJECT_ROOT / 'minimum_set' / 'test_test' / 'output' / 'PMC12627845_known_facts_minimal_references.json'

RUN_DATA_PREPARATION = True
RUN_CONTEXT_EXTRACTION = True
RUN_STAGE1_VISION = False
RUN_STAGE2_FUSION = False
RUN_STAGE3_REASONING = False
FORCE_REBUILD = False

TEXT_MODEL = 'deepseek-v4-pro'
TEXT_BATCH_DELAY_SECONDS = 0.5

for folder in [DATA_ROOT, DATA_ROOT / 'references', DATA_ROOT / 'provenance',
               DATA_ROOT / 'reasoning_inputs', DATA_ROOT / 'reasoning_outputs',
               DATA_ROOT / 'sealed_target']:
    folder.mkdir(parents=True, exist_ok=True)

print('Data root:', DATA_ROOT)
print('Source result exists:', SOURCE_RESULT.exists())

Data root: C:\Users\sxx\Desktop\codex\barchmark-m-7.30\LLM_Reasoning\data\test
Source result exists: True


## 第2课：四个视觉模型如何配置

Stage 1要求四个真正支持图片输入的模型。DeepSeek文本模型不能代替视觉模型。本教程使用OpenAI兼容接口格式，但不把密钥写进Notebook。

请在下面四个槽位填写你实际拥有的视觉模型。每个槽位从指定环境变量读取密钥。只要接口兼容OpenAI Chat Completions，就无需改后续代码。

为了防止把同一个模型调用四次伪装成四模型，代码会检查四个 `model` 名称是否互不相同。

In [2]:
VISION_EXPERTS = [
    {
        'expert_id': 'vision_expert_1',
        'base_url': 'https://api.openai.com/v1',
        'model': 'REPLACE_WITH_VISION_MODEL_1',
        'api_key_env': 'VISION_API_KEY_1',
    },
    {
        'expert_id': 'vision_expert_2',
        'base_url': 'REPLACE_WITH_OPENAI_COMPATIBLE_BASE_URL_2',
        'model': 'REPLACE_WITH_VISION_MODEL_2',
        'api_key_env': 'VISION_API_KEY_2',
    },
    {
        'expert_id': 'vision_expert_3',
        'base_url': 'REPLACE_WITH_OPENAI_COMPATIBLE_BASE_URL_3',
        'model': 'REPLACE_WITH_VISION_MODEL_3',
        'api_key_env': 'VISION_API_KEY_3',
    },
    {
        'expert_id': 'vision_expert_4',
        'base_url': 'REPLACE_WITH_OPENAI_COMPATIBLE_BASE_URL_4',
        'model': 'REPLACE_WITH_VISION_MODEL_4',
        'api_key_env': 'VISION_API_KEY_4',
    },
]

def validate_vision_experts(experts):
    if len(experts) != 4:
        raise ValueError('Exactly four vision experts are required.')
    models = [item['model'] for item in experts]
    if len(set(models)) != 4:
        raise ValueError('The four vision model names must be different.')
    placeholders = [value for item in experts for value in (item['base_url'], item['model'])
                    if 'REPLACE_WITH' in value]
    if placeholders:
        raise ValueError('Replace all vision provider placeholders before Stage 1.')
    missing_keys = [item['api_key_env'] for item in experts if not os.getenv(item['api_key_env'], '').strip()]
    if missing_keys:
        raise RuntimeError('Missing environment variables: ' + ', '.join(missing_keys))

if RUN_STAGE1_VISION:
    validate_vision_experts(VISION_EXPERTS)
else:
    print('Stage 1 is currently off; provider placeholders do not block data preparation.')

Stage 1 is currently off; provider placeholders do not block data preparation.


## 第3课：准备一些基础工具函数

In [3]:
XLINK = '{http://www.w3.org/1999/xlink}href'
OA_API = 'https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi'

def normalize_text(node):
    if node is None:
        return ''
    return html.unescape(' '.join(''.join(node.itertext()).split()))

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

def safe_name(value):
    value = re.sub(r'[^A-Za-z0-9_.-]+', '_', value or '').strip('._')
    return value or 'unnamed'

def read_api_key(paths, env_name):
    value = os.getenv(env_name, '').strip()
    if value:
        return value
    for path in paths:
        path = Path(path)
        if path.exists() and path.read_text(encoding='utf-8').strip():
            return path.read_text(encoding='utf-8').strip()
    return ''

## 第4课：下载PMC开放获取原文包

PMC开放获取包通常包含JATS XML和论文图片。我们下载整个压缩包，而不是截图网页，因此可以保留原始图片文件和可追溯的图注。

In [4]:
def http_get_with_retries(url, params=None, attempts=5):
    last_error = None
    for attempt in range(attempts):
        try:
            return httpx.get(url, params=params, timeout=180, follow_redirects=True)
        except httpx.HTTPError as exc:
            last_error = exc
            if attempt < attempts - 1:
                time.sleep(min(2 ** attempt, 16))
    raise last_error

def download_oa_package(pmcid, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    package_path = destination / f'{pmcid}.tar.gz'
    if package_path.exists() and package_path.stat().st_size > 0 and not FORCE_REBUILD:
        return package_path
    try:
        response = http_get_with_retries(OA_API, params={'id': pmcid})
        response.raise_for_status()
        root = ET.fromstring(response.content)
        link = root.find(".//link[@format='tgz']")
        if link is None:
            return None
        url = link.attrib['href'].replace('ftp://', 'https://')
        package_response = http_get_with_retries(url)
        if package_response.status_code >= 400:
            return None
        package_path.write_bytes(package_response.content)
        return package_path
    except (httpx.HTTPError, ET.ParseError):
        return None

def fetch_pmc_xml_fallback(pmcid, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not FORCE_REBUILD:
        return destination
    numeric_id = pmcid.upper().removeprefix('PMC')
    url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
    response = http_get_with_retries(url, params={'db': 'pmc', 'id': numeric_id, 'retmode': 'xml'})
    response.raise_for_status()
    destination.write_bytes(response.content)
    return destination

def download_graphic_fallback(pmcid, href, destination, html_cache):
    destination = Path(destination)
    if destination.exists() and not FORCE_REBUILD:
        return destination
    html_cache = Path(html_cache)
    if not html_cache.exists() or FORCE_REBUILD:
        response = http_get_with_retries(f'https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}/')
        response.raise_for_status()
        html_cache.write_text(response.text, encoding='utf-8')
    page = html_cache.read_text(encoding='utf-8')
    basename = Path(href).name
    pattern = r'src="([^"]*' + re.escape(basename) + r'[^"]*)"'
    matches = re.findall(pattern, page, flags=re.IGNORECASE)
    if not matches:
        return None
    url = html.unescape(matches[0])
    if url.startswith('//'):
        url = 'https:' + url
    elif url.startswith('/'):
        url = 'https://pmc.ncbi.nlm.nih.gov' + url
    response = http_get_with_retries(url)
    response.raise_for_status()
    content_type = response.headers.get('content-type', '')
    if 'image' not in content_type.lower():
        raise RuntimeError(f'Fallback URL did not return an image: {url}')
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(response.content)
    return destination

def safe_extract_tar(package_path, extraction_dir):
    extraction_dir = Path(extraction_dir).resolve()
    extraction_dir.mkdir(parents=True, exist_ok=True)
    marker = extraction_dir / '.extracted'
    if marker.exists() and not FORCE_REBUILD:
        return
    with tarfile.open(package_path, 'r:gz') as archive:
        safe_members = []
        for member in archive.getmembers():
            target = (extraction_dir / member.name).resolve()
            if extraction_dir not in target.parents and target != extraction_dir:
                raise RuntimeError(f'Unsafe archive member: {member.name}')
            safe_members.append(member)
        archive.extractall(extraction_dir, members=safe_members)
    marker.write_text('ok', encoding='utf-8')

def find_article_xml(extraction_dir):
    candidates = [path for path in Path(extraction_dir).rglob('*.xml') if path.is_file()]
    for path in candidates:
        try:
            root = ET.parse(path).getroot()
            if root.find('.//article-meta') is not None:
                return path
        except ET.ParseError:
            continue
    raise FileNotFoundError(f'No JATS article XML found in {extraction_dir}')

## 第5课：把一篇论文整理为独立素材包

这个函数会保存原文XML、全文纯文本、论文元数据、所有JATS `<fig>`节点对应的图片和图注。每张图片都获得稳定的 `figure_uid`，后续任何模型输出都用它关联，避免把不同论文或不同图片混在一起。

In [5]:
def locate_graphic_file(extraction_dir, href):
    href_path = Path(href)
    exact = list(Path(extraction_dir).rglob(href_path.name))
    if exact:
        return exact[0]
    stem_matches = [path for path in Path(extraction_dir).rglob('*')
                    if path.is_file() and path.stem.lower() == href_path.stem.lower()]
    return stem_matches[0] if stem_matches else None

def prepare_reference_package(ref_number, selected_reference):
    citation = selected_reference['citation']
    ref_id = selected_reference['ref_id']
    pmcid = citation['pmcid']
    package_id = f'ref{ref_number}_{ref_id}_{pmcid}'
    package_dir = DATA_ROOT / 'references' / package_id
    source_dir = package_dir / 'source'
    figures_dir = package_dir / 'figures'
    raw_dir = package_dir / '_oa_package'
    for folder in [source_dir, figures_dir, package_dir / 'derived' / 'observations']:
        folder.mkdir(parents=True, exist_ok=True)

    archive = download_oa_package(pmcid, raw_dir)
    extracted = raw_dir / 'extracted'
    if archive is not None:
        safe_extract_tar(archive, extracted)
        xml_path = find_article_xml(extracted)
        source_mode = 'pmc_oa_package'
    else:
        extracted.mkdir(parents=True, exist_ok=True)
        xml_path = fetch_pmc_xml_fallback(pmcid, extracted / f'{pmcid}.xml')
        source_mode = 'pmc_efetch_with_html_image_fallback'
    root = ET.parse(xml_path).getroot()
    article_xml = source_dir / 'article.xml'
    shutil.copy2(xml_path, article_xml)

    title = normalize_text(root.find('.//article-meta/title-group/article-title'))
    abstract = normalize_text(root.find('.//article-meta/abstract'))
    body = root.find('.//article/body')
    full_text = normalize_text(body)
    (source_dir / 'full_text.txt').write_text(full_text, encoding='utf-8')
    article_meta = {
        'package_id': package_id, 'reference_number': ref_number, 'ref_id': ref_id,
        'pmcid': pmcid, 'pmid': citation.get('pmid', ''), 'doi': citation.get('doi', ''),
        'title': title or citation.get('title', ''), 'abstract': abstract,
        'citation_text': citation.get('citation_text', ''),
        'directly_supported_fact_ids': selected_reference.get('directly_supported_fact_ids', []),
        'source_xml_sha256': sha256_file(article_xml), 'source_mode': source_mode,
    }
    write_json(source_dir / 'article.json', article_meta)

    figure_records = []
    seen_source_files = set()
    for figure_index, fig in enumerate(root.findall('.//fig'), 1):
        fig_id = fig.attrib.get('id', '') or f'fig{figure_index}'
        label = normalize_text(fig.find('./label')) or f'Figure {figure_index}'
        caption = normalize_text(fig.find('./caption'))
        graphics = fig.findall('.//graphic') + fig.findall('.//inline-graphic')
        for asset_index, graphic in enumerate(graphics, 1):
            href = graphic.attrib.get(XLINK, '') or graphic.attrib.get('href', '')
            if not href:
                continue
            source_asset = locate_graphic_file(extracted, href)
            source_identity = source_asset.resolve() if source_asset is not None else f'fallback:{href}'
            if source_identity in seen_source_files:
                continue
            seen_source_files.add(source_identity)
            suffix = Path(href).suffix.lower() or (source_asset.suffix.lower() if source_asset else '.jpg')
            figure_uid = f'{package_id}__{safe_name(fig_id)}__asset{asset_index}'
            output_asset = figures_dir / f'{figure_uid}{suffix}'
            if source_asset is not None:
                shutil.copy2(source_asset, output_asset)
            else:
                downloaded = download_graphic_fallback(
                    pmcid, href, output_asset, raw_dir / f'{pmcid}_article.html')
                if downloaded is None:
                    continue
            figure_records.append({
                'figure_uid': figure_uid, 'figure_id_in_article': fig_id,
                'figure_label': label, 'asset_index': asset_index, 'caption': caption,
                'image_path': output_asset.relative_to(DATA_ROOT).as_posix(),
                'source_href': href, 'mime_type': mimetypes.guess_type(output_asset.name)[0] or 'application/octet-stream',
                'sha256': sha256_file(output_asset),
            })
    write_json(package_dir / 'figures_manifest.json', figure_records)
    return {
        **article_meta, 'package_path': package_dir.relative_to(DATA_ROOT).as_posix(),
        'full_text_path': (source_dir / 'full_text.txt').relative_to(DATA_ROOT).as_posix(),
        'article_xml_path': article_xml.relative_to(DATA_ROOT).as_posix(),
        'figures_manifest_path': (package_dir / 'figures_manifest.json').relative_to(DATA_ROOT).as_posix(),
        'figure_count': len(figure_records),
    }

## 第6课：建立总索引并封存目标假说

这里读取上一阶段JSON中的最小参考文献集合。`sealed_target`保存真实Gap/Hypothesis，但总索引中的 `allowed_for_stage1_to_stage3=false` 明确禁止下游读取。

In [6]:
def prepare_dataset():
    source_result = json.loads(SOURCE_RESULT.read_text(encoding='utf-8'))
    provenance_copy = DATA_ROOT / 'provenance' / SOURCE_RESULT.name
    shutil.copy2(SOURCE_RESULT, provenance_copy)

    target = {
        'target_paper': source_result['target_paper'],
        'knowledge_gap': source_result['minimum_reasoning_chain']['knowledge_gap'],
        'hypothesis': source_result['minimum_reasoning_chain']['hypothesis'],
        'access_policy': {
            'allowed_for_stage1_to_stage3': False,
            'allowed_use': 'post-hoc human evaluation only',
            'reason': 'Prevent answer leakage into independent hypothesis generation.',
        },
    }
    sealed_path = DATA_ROOT / 'sealed_target' / 'target_hypothesis.json'
    write_json(sealed_path, target)

    reference_packages = []
    for number, selected_reference in enumerate(source_result['minimum_reference_set']['selected_references'], 1):
        print(f'Preparing reference {number}...')
        reference_packages.append(prepare_reference_package(number, selected_reference))

    manifest = {
        'schema_version': 'visual-reasoning-dataset-1.0',
        'dataset_id': 'PMC12627845_minimal_reference_visual_reasoning',
        'source_reasoning_result': provenance_copy.relative_to(DATA_ROOT).as_posix(),
        'reference_count': len(reference_packages), 'references': reference_packages,
        'sealed_target_path': sealed_path.relative_to(DATA_ROOT).as_posix(),
        'leakage_firewall': {
            'stage1_allowed_inputs': ['one image', 'that image caption'],
            'stage2_allowed_inputs': ['four descriptions of the same image'],
            'context_allowed_inputs': ['one reference full text', 'its figure captions'],
            'stage3_allowed_inputs': ['each reference context', 'its standardized observations'],
            'forbidden_for_all_generation_stages': ['sealed_target/**', 'target paper results', 'target paper conclusion'],
        },
    }
    write_json(DATA_ROOT / 'dataset_manifest.json', manifest)
    return manifest

if RUN_DATA_PREPARATION:
    dataset_manifest = prepare_dataset()
else:
    dataset_manifest = json.loads((DATA_ROOT / 'dataset_manifest.json').read_text(encoding='utf-8'))

print('References:', dataset_manifest['reference_count'])
for item in dataset_manifest['references']:
    print(item['package_id'], 'figures =', item['figure_count'])

Preparing reference 1...
Preparing reference 2...
Preparing reference 3...
References: 3
ref1_CR22_PMC7612910 figures = 15
ref2_CR23_PMC6746329 figures = 5
ref3_CR38_PMC7103375 figures = 4


## 第7课：定义三个阶段的英文Prompt

In [7]:
STAGE1_VISION_PROMPT = r'''You are a professional cell-biology image observer. Analyze only the single supplied image and its dedicated caption. Describe only directly visible visual features.

Allowed content includes color or staining intensity, brightness, shape, size, boundaries, spatial distribution, layering, arrangement, density, texture, visible labels, axes, bars, symbols, and relative visual differences between explicitly labeled regions or panels.

Forbidden content:
- biological inference, mechanism, causal explanation, experimental interpretation, sub-conclusion, or final conclusion;
- the goal of the experiment;
- claims that the image suggests, indicates, demonstrates, proves, implies, reveals, or predicts anything;
- information from another image, another paper, or outside knowledge.

Return exactly one coherent paragraph of objective visual description. Do not add a heading or mention these instructions.'''

STAGE2_FUSION_PROMPT = r'''You receive four independent descriptions of the same single image. Produce the one standardized Observation for this image.

Rules:
- Process only this image. Never import another image or paper.
- Preserve visual features independently reported by multiple models.
- Keep a detail unique to one model only when it does not conflict with any other description and is phrased as directly visible.
- Delete contradictory details rather than choosing a side.
- Delete biological interpretation, mechanism, speculation, experimental goals, sub-conclusions, and conclusions.
- Do not use inferential verbs such as suggest, indicate, demonstrate, prove, imply, reveal, or predict.

Return JSON only: {"observation": "one coherent canonical Observation paragraph"}.'''

CONTEXT_PROMPT = r'''You are a cell-biology literature extraction specialist. Build exactly one global experimental context for the supplied reference paper. Use only the supplied paper text and figure captions.

Include only information needed to understand this paper's experimental image set:
- the core scientific question;
- all experimental groups and their control/treatment relationships;
- sample types, organisms, cells, tissues, and relevant biological materials;
- staining, imaging, measurement, or detection methods;
- treatment conditions, variables, doses, times, and perturbations when present;
- the paper's original textual observations;
- which figure belongs to which comparison, control, treatment, or assay.

Exclude unrelated review background, unrelated experiments, discussion extensions, outside knowledge, new inference, mechanism speculation, and all visual descriptions derived from looking at images. Do not mix information from another paper.

Return JSON only:
{
  "context": "one concise but complete global context paragraph",
  "figure_experiment_map": [
    {"figure_id": "article figure ID", "groups": ["control", "treatment"], "sample": "sample type", "method": "method", "conditions": "conditions", "original_text_observation": "paper-stated observation only"}
  ],
  "warnings": []
}'''

STAGE3_REASONING_PROMPT = r'''You are a cell-biology researcher. Generate multiple scientifically reasonable hypotheses using only the supplied minimal prior-reference evidence packages. Each package contains one paper-specific global context and standardized image Observations.

Hard constraints:
1. First produce exactly one local proposition per reference paper. A proposition may integrate that paper's context and Observations, but must remain an objectively established prior-paper fact. Do not make cross-paper links at this layer.
2. Then form multiple cross-paper combinations. Each combination must use exactly one proposition from every reference paper. Represent the combination only as an ordered list of proposition IDs; do not reveal hidden chain-of-thought or narrative intermediate reasoning.
3. For each combination, state one specific untested relationship as a Knowledge Gap and one falsifiable Hypothesis that answers it.
4. Generate 2 to 6 Gap–Hypothesis pairs. Different pairs need not have the same structure.
5. Never provide an experiment goal, raw visual phenomenon, experiment interpretation, sub-conclusion, final conclusion, target-paper result, or target-paper answer.
6. Do not mention or assume any information not present in the supplied reference packages.

Return JSON only:
{
  "local_propositions": [
    {"proposition_id": "P1", "reference_package_id": "ref1...", "statement": "one established prior-paper fact", "supporting_observation_ids": ["figure_uid"]}
  ],
  "hypothesis_set": [
    {"gap_id": "G1", "proposition_path": ["P1", "P2", "P3"], "knowledge_gap": "one untested relationship", "hypothesis_id": "H1", "hypothesis": "one falsifiable answer", "testable_variables": {"independent": "variable", "dependent": "variable"}}
  ],
  "warnings": []
}'''

## 第8课：统一的模型调用与缓存

所有结果先写缓存再进入下一阶段。这样网络中断后可以续跑，也不会重复付费。视觉模型收到图片的Base64数据和该图图注；文本模型只收到允许的文字字段。

In [8]:
def image_data_url(path):
    path = Path(path)
    mime = mimetypes.guess_type(path.name)[0] or 'image/png'
    encoded = base64.b64encode(path.read_bytes()).decode('ascii')
    return f'data:{mime};base64,{encoded}'

def call_openai_compatible(provider, messages, cache_path, json_output=False):
    cache_path = Path(cache_path)
    if cache_path.exists() and not FORCE_REBUILD:
        return json.loads(cache_path.read_text(encoding='utf-8'))
    from openai import OpenAI
    key = os.getenv(provider['api_key_env'], '').strip()
    if not key:
        raise RuntimeError(f"Missing {provider['api_key_env']}")
    client = OpenAI(api_key=key, base_url=provider['base_url'])
    kwargs = {'model': provider['model'], 'messages': messages, 'max_tokens': 8192, 'stream': False}
    if json_output:
        kwargs['response_format'] = {'type': 'json_object'}
    response = client.chat.completions.create(**kwargs)
    result = {
        'provider': provider['expert_id'], 'model': provider['model'],
        'content': (response.choices[0].message.content or '').strip(),
    }
    write_json(cache_path, result)
    return result

def deepseek_provider():
    key_paths = [
        PROJECT_ROOT / 'minimum_set' / 'key' / 'DEEPSEEK_API_KEY.txt',
        PROJECT_ROOT / 'code' / 'key' / 'DEEPSEEK_API_KEY.txt',
    ]
    key = read_api_key(key_paths, 'DEEPSEEK_API_KEY')
    if not key:
        raise RuntimeError('DeepSeek API key was not found.')
    return {'expert_id': 'deepseek_text', 'base_url': 'https://api.deepseek.com',
            'model': TEXT_MODEL, 'api_key_env': '_INTERNAL_DEEPSEEK_KEY'}, key

def call_deepseek_json(system_prompt, payload, cache_path):
    cache_path = Path(cache_path)
    if cache_path.exists() and not FORCE_REBUILD:
        return json.loads(cache_path.read_text(encoding='utf-8'))
    from openai import OpenAI
    provider, key = deepseek_provider()
    client = OpenAI(api_key=key, base_url=provider['base_url'])
    last_error = None
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=provider['model'],
                messages=[{'role': 'system', 'content': system_prompt},
                          {'role': 'user', 'content': json.dumps(payload, ensure_ascii=False)}],
                max_tokens=32768, reasoning_effort='high',
                extra_body={'thinking': {'type': 'enabled'}},
                response_format={'type': 'json_object'}, stream=False)
            value = json.loads((response.choices[0].message.content or '').strip())
            write_json(cache_path, value)
            return value
        except Exception as exc:
            last_error = exc
            if attempt < 2:
                time.sleep(2 ** attempt)
    raise RuntimeError(f'DeepSeek call failed after 3 attempts: {last_error}')

## 第9课：每篇文献只生成一次全局Context

In [9]:
def build_all_contexts(manifest):
    contexts = []
    for reference in manifest['references']:
        package_dir = DATA_ROOT / reference['package_path']
        context_path = package_dir / 'derived' / 'context.json'
        full_text = (DATA_ROOT / reference['full_text_path']).read_text(encoding='utf-8')
        figures = json.loads((DATA_ROOT / reference['figures_manifest_path']).read_text(encoding='utf-8'))
        payload = {
            'reference_package_id': reference['package_id'],
            'paper_title': reference['title'],
            'full_paper_text': full_text,
            'figure_captions': [{
                'figure_id': item['figure_id_in_article'],
                'figure_label': item['figure_label'], 'caption': item['caption']
            } for item in figures],
        }
        context = call_deepseek_json(CONTEXT_PROMPT, payload, context_path)
        contexts.append({'reference_package_id': reference['package_id'], 'context_path': context_path, 'context': context})
        time.sleep(TEXT_BATCH_DELAY_SECONDS)
    return contexts

if RUN_CONTEXT_EXTRACTION:
    context_results = build_all_contexts(dataset_manifest)
    print('Context files:', len(context_results))
else:
    print('Context extraction is off. Set RUN_CONTEXT_EXTRACTION=True when ready.')

Context files: 3


## 第10课：Stage 1——四个视觉模型逐图独立描述

In [10]:
def run_stage1(manifest):
    validate_vision_experts(VISION_EXPERTS)
    completed = []
    for reference in manifest['references']:
        package_dir = DATA_ROOT / reference['package_path']
        figures = json.loads((DATA_ROOT / reference['figures_manifest_path']).read_text(encoding='utf-8'))
        for figure in figures:
            figure_dir = package_dir / 'derived' / 'observations' / figure['figure_uid']
            expert_dir = figure_dir / 'expert_descriptions'
            expert_dir.mkdir(parents=True, exist_ok=True)
            image_url = image_data_url(DATA_ROOT / figure['image_path'])
            for provider in VISION_EXPERTS:
                messages = [
                    {'role': 'system', 'content': STAGE1_VISION_PROMPT},
                    {'role': 'user', 'content': [
                        {'type': 'text', 'text': 'Dedicated caption:\n' + figure['caption']},
                        {'type': 'image_url', 'image_url': {'url': image_url, 'detail': 'high'}},
                    ]},
                ]
                result = call_openai_compatible(
                    provider, messages, expert_dir / f"{provider['expert_id']}.json")
                completed.append((figure['figure_uid'], provider['expert_id']))
    return completed

if RUN_STAGE1_VISION:
    stage1_completed = run_stage1(dataset_manifest)
    print('Stage 1 expert outputs:', len(stage1_completed))
else:
    print('Stage 1 is off. Configure four vision providers, then set RUN_STAGE1_VISION=True.')

Stage 1 is off. Configure four vision providers, then set RUN_STAGE1_VISION=True.


## 第11课：Stage 2——同一张图的四专家融合

In [11]:
def run_stage2(manifest):
    observations = []
    for reference in manifest['references']:
        package_dir = DATA_ROOT / reference['package_path']
        figures = json.loads((DATA_ROOT / reference['figures_manifest_path']).read_text(encoding='utf-8'))
        for figure in figures:
            figure_dir = package_dir / 'derived' / 'observations' / figure['figure_uid']
            descriptions = []
            for provider in VISION_EXPERTS:
                path = figure_dir / 'expert_descriptions' / f"{provider['expert_id']}.json"
                if not path.exists():
                    raise FileNotFoundError(f'Missing Stage 1 output: {path}')
                item = json.loads(path.read_text(encoding='utf-8'))
                descriptions.append({'expert_id': provider['expert_id'], 'model': item['model'], 'description': item['content']})
            payload = {'figure_uid': figure['figure_uid'], 'descriptions_of_this_image_only': descriptions}
            fusion = call_deepseek_json(STAGE2_FUSION_PROMPT, payload, figure_dir / 'observation.json')
            if isinstance(fusion, dict) and 'observation' not in fusion:
                fusion = {'observation': next(iter(fusion.values())) if fusion else '', 'raw': fusion}
                write_json(figure_dir / 'observation.json', fusion)
            observations.append({'figure_uid': figure['figure_uid'], 'observation': fusion.get('observation', '')})
    return observations

if RUN_STAGE2_FUSION:
    stage2_observations = run_stage2(dataset_manifest)
    print('Canonical observations:', len(stage2_observations))
else:
    print('Stage 2 is off. It requires all four Stage 1 descriptions for every image.')

Stage 2 is off. It requires all four Stage 1 descriptions for every image.


## 第12课：构造Stage 3唯一允许读取的输入

这个单元不会读取 `sealed_target`。它只收集三篇最小参考文献各自的Context和标准Observation。这样可以从代码层面证明目标假说没有泄漏。

In [12]:
def build_stage3_input(manifest):
    packages = []
    for reference in manifest['references']:
        package_dir = DATA_ROOT / reference['package_path']
        context_path = package_dir / 'derived' / 'context.json'
        if not context_path.exists():
            raise FileNotFoundError(f'Missing context: {context_path}')
        figures = json.loads((DATA_ROOT / reference['figures_manifest_path']).read_text(encoding='utf-8'))
        observations = []
        for figure in figures:
            observation_path = package_dir / 'derived' / 'observations' / figure['figure_uid'] / 'observation.json'
            if not observation_path.exists():
                raise FileNotFoundError(f'Missing observation: {observation_path}')
            value = json.loads(observation_path.read_text(encoding='utf-8'))
            observations.append({
                'observation_id': figure['figure_uid'],
                'figure_id_in_article': figure['figure_id_in_article'],
                'observation': value['observation'],
            })
        packages.append({
            'reference_package_id': reference['package_id'],
            'context': json.loads(context_path.read_text(encoding='utf-8')),
            'standardized_observations': observations,
        })
    stage3_input = {
        'schema_version': 'stage3-input-1.0',
        'minimal_reference_packages': packages,
        'explicitly_excluded_paths': ['sealed_target/**', 'provenance/**'],
    }
    output_path = DATA_ROOT / 'reasoning_inputs' / 'stage3_input.json'
    write_json(output_path, stage3_input)
    if 'target_paper' in stage3_input or 'target_hypothesis' in stage3_input:
        raise RuntimeError('Potential target-answer leakage detected in Stage 3 input.')
    return stage3_input

if RUN_STAGE3_REASONING:
    stage3_input = build_stage3_input(dataset_manifest)
else:
    print('Stage 3 input will be built only after Context and all Observations exist.')

Stage 3 input will be built only after Context and all Observations exist.


## 第13课：Stage 3——多文献联合生成多个Gap与Hypothesis

In [13]:
def validate_stage3_output(result, reference_count):
    propositions = result.get('local_propositions', [])
    hypotheses = result.get('hypothesis_set', [])
    if len(propositions) != reference_count:
        raise ValueError('Stage 3 must produce exactly one proposition per reference paper.')
    proposition_ids = {item.get('proposition_id') for item in propositions}
    if len(proposition_ids) != reference_count:
        raise ValueError('Proposition IDs must be unique.')
    if not 2 <= len(hypotheses) <= 6:
        raise ValueError('Stage 3 must produce 2–6 Gap–Hypothesis pairs.')
    for item in hypotheses:
        if set(item.get('proposition_path', [])) != proposition_ids:
            raise ValueError(f"{item.get('gap_id')} must use one proposition from every paper.")
        if not item.get('knowledge_gap') or not item.get('hypothesis'):
            raise ValueError('Every item requires a gap and a hypothesis.')
    return result

if RUN_STAGE3_REASONING:
    stage3_result_path = DATA_ROOT / 'reasoning_outputs' / 'stage3_hypotheses.json'
    stage3_result = call_deepseek_json(
        STAGE3_REASONING_PROMPT, stage3_input, stage3_result_path)
    validate_stage3_output(stage3_result, dataset_manifest['reference_count'])
    print('Generated hypotheses:', len(stage3_result['hypothesis_set']))
    print('Saved:', stage3_result_path)
else:
    print('Stage 3 reasoning is off.')

Stage 3 reasoning is off.


## 第14课：最后检查，而不是把答案喂给模型

只有Stage 3结束后，人类研究者才可以分别打开：

- `reasoning_outputs/stage3_hypotheses.json`：模型独立生成的假说；
- `sealed_target/target_hypothesis.json`：目标论文真实假说。

比较两者属于评价阶段，不属于生成阶段。本Notebook故意不自动把两者同时发送给任何模型。

In [14]:
def audit_dataset(manifest):
    rows = []
    for reference in manifest['references']:
        figures = json.loads((DATA_ROOT / reference['figures_manifest_path']).read_text(encoding='utf-8'))
        missing_images = [item['image_path'] for item in figures if not (DATA_ROOT / item['image_path']).exists()]
        rows.append({
            'package_id': reference['package_id'], 'figure_count': len(figures),
            'missing_images': missing_images,
            'xml_exists': (DATA_ROOT / reference['article_xml_path']).exists(),
            'full_text_exists': (DATA_ROOT / reference['full_text_path']).exists(),
        })
    sealed = DATA_ROOT / manifest['sealed_target_path']
    return {
        'references': rows, 'sealed_target_exists': sealed.exists(),
        'all_source_material_complete': all(
            row['xml_exists'] and row['full_text_exists'] and not row['missing_images'] for row in rows),
    }

audit = audit_dataset(dataset_manifest)
print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "references": [
    {
      "package_id": "ref1_CR22_PMC7612910",
      "figure_count": 15,
      "missing_images": [],
      "xml_exists": true,
      "full_text_exists": true
    },
    {
      "package_id": "ref2_CR23_PMC6746329",
      "figure_count": 5,
      "missing_images": [],
      "xml_exists": true,
      "full_text_exists": true
    },
    {
      "package_id": "ref3_CR38_PMC7103375",
      "figure_count": 4,
      "missing_images": [],
      "xml_exists": true,
      "full_text_exists": true
    }
  ],
  "sealed_target_exists": true,
  "all_source_material_complete": true
}
